In [0]:
# List all tables in the rearc schema
tables_df = spark.sql("""
    SHOW TABLES IN workspace.rearc
""")

display(tables_df)

In [0]:
q1_sql = spark.sql("""
    SELECT
        COUNT(*) AS total_years,
        MIN(year) AS start_year,
        MAX(year) AS end_year,
        ROUND(AVG(population), 2) AS mean_population,
        ROUND(STDDEV_SAMP(population), 2) AS stddev_population
    FROM workspace.rearc.silver_datausa_population
    WHERE year BETWEEN 2013 AND 2018
""")

display(q1_sql)

In [0]:
from pyspark.sql import functions as F

population_df = spark.table(
    "workspace.rearc.silver_datausa_population"
)

q1_pyspark = (
    population_df
    .filter(F.col("year").between(2013, 2018))
    .agg(
        F.count("*").alias("total_years"),
        F.min("year").alias("start_year"),
        F.max("year").alias("end_year"),
        F.round(F.avg("population"), 2).alias("mean_population"),
        F.round(F.stddev_samp("population"), 2).alias("stddev_population")
    )
)

display(q1_pyspark)

In [0]:
(
    q1_pyspark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.rearc.gold_q1_population_stats")
)

print("GOLD Q1 TABLE CREATED SUCCESSFULLY")

In [0]:
series_df = spark.table("workspace.rearc.silver_bls_series")

print("SERIES TABLE SCHEMA")
series_df.printSchema()

display(series_df.limit(5))

In [0]:
q2_sql = spark.sql("""
WITH quarterly_totals AS (
    SELECT
        TRIM(series_id) AS series_id,
        CAST(year AS INT) AS year,
        SUM(CAST(value AS DOUBLE)) AS annual_quarterly_total
    FROM workspace.rearc.silver_bls_data
    WHERE period IN ('Q01', 'Q02', 'Q03', 'Q04')
      AND value IS NOT NULL
    GROUP BY TRIM(series_id), CAST(year AS INT)
),

ranked_years AS (
    SELECT
        series_id,
        year,
        annual_quarterly_total,
        ROW_NUMBER() OVER (
            PARTITION BY series_id
            ORDER BY annual_quarterly_total DESC, year ASC
        ) AS year_rank
    FROM quarterly_totals
)

SELECT
    r.series_id,
    CONCAT_WS(
        ' | ',
        s.sector_name,
        s.class_text,
        s.measure_text,
        s.duration_text,
        s.seasonal_text
    ) AS series_label,
    r.year AS highest_year,
    ROUND(r.annual_quarterly_total, 2) AS highest_quarterly_total
FROM ranked_years r
INNER JOIN workspace.rearc.silver_bls_series s
    ON r.series_id = TRIM(s.series_id)
WHERE r.year_rank = 1
ORDER BY r.series_id
""")

display(q2_sql)

In [0]:
q2_sql.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.rearc.gold_q2_productivity")

In [0]:
spark.table("workspace.rearc.gold_q2_productivity").count()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Read Silver tables
bls_df = spark.table("workspace.rearc.silver_bls_data")
series_df = spark.table("workspace.rearc.silver_bls_series")

# 2. Calculate yearly total using only Q01-Q04
quarterly_totals = (
    bls_df
    .filter(
        F.col("period").isin("Q01", "Q02", "Q03", "Q04")
        & F.col("value").isNotNull()
    )
    .groupBy(
        F.trim("series_id").alias("series_id"),
        F.col("year").cast("int").alias("year")
    )
    .agg(
        F.sum(F.col("value").cast("double")).alias(
            "quarterly_total"
        )
    )
)

# 3. Rank years within each series
window_spec = Window.partitionBy("series_id").orderBy(
    F.desc("quarterly_total"),
    F.asc("year")
)

ranked_df = (
    quarterly_totals
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .alias("r")
)

# 4. Prepare series metadata with a unique set of columns
metadata_df = (
    series_df
    .select(
        F.trim("series_id").alias("metadata_series_id"),
        F.concat_ws(
            " | ",
            "sector_name",
            "class_text",
            "measure_text",
            "duration_text",
            "seasonal_text"
        ).alias("series_label")
    )
    .alias("m")
)

# 5. Join using explicit aliases
q2_pyspark = (
    ranked_df
    .join(
        metadata_df,
        F.col("r.series_id") == F.col("m.metadata_series_id"),
        "inner"
    )
    .select(
        F.col("r.series_id"),
        F.col("m.series_label"),
        F.col("r.year").alias("highest_year"),
        F.round(F.col("r.quarterly_total"), 2).alias(
            "highest_quarterly_total"
        )
    )
    .orderBy("series_id")
)

display(q2_pyspark)

In [0]:
q3_sql = spark.sql("""
    SELECT
        TRIM(b.series_id) AS series_id,
        CONCAT_WS(
            ' | ',
            s.sector_name,
            s.class_text,
            s.measure_text,
            s.duration_text,
            s.seasonal_text
        ) AS series_label,
        b.year,
        b.period,
        b.value AS annual_value,
        p.population
    FROM workspace.rearc.silver_bls_data b
    LEFT JOIN workspace.rearc.silver_bls_series s
        ON TRIM(b.series_id) = TRIM(s.series_id)
    LEFT JOIN workspace.rearc.silver_datausa_population p
        ON b.year = p.year
    WHERE TRIM(b.series_id) = 'PRS30006032'
      AND b.period = 'Q01'
    ORDER BY b.year
""")

display(q3_sql)

In [0]:
q3_sql.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.rearc.gold_q3_bls_population")

print("GOLD Q3 TABLE CREATED SUCCESSFULLY")

print(
    "Total records:",
    spark.table("workspace.rearc.gold_q3_bls_population").count()
)

In [0]:
from pyspark.sql import functions as F

# Load Silver tables
bls_df = spark.table("workspace.rearc.silver_bls_data")
series_df = spark.table("workspace.rearc.silver_bls_series")
population_df = spark.table("workspace.rearc.silver_datausa_population")

# Build readable series label
series_label_df = series_df.select(
    F.trim("series_id").alias("series_id"),
    F.concat_ws(
        " | ",
        "sector_name",
        "class_text",
        "measure_text",
        "duration_text",
        "seasonal_text"
    ).alias("series_label")
)

# Filter BLS records
bls_q3_df = (
    bls_df
    .filter(
        (F.trim("series_id") == "PRS30006032") &
        (F.col("period") == "Q01")
    )
    .select(
        F.trim("series_id").alias("series_id"),
        "year",
        "period",
        F.col("value").cast("double").alias("annual_value")
    )
)

# Join series metadata and population
q3_pyspark = (
    bls_q3_df
    .join(series_label_df, on="series_id", how="left")
    .join(
        population_df.select(
            "year",
            F.col("population").cast("double").alias("population")
        ),
        on="year",
        how="left"
    )
    .select(
        "series_id",
        "series_label",
        "year",
        "period",
        "annual_value",
        "population"
    )
    .orderBy("year")
)

display(q3_pyspark)

In [0]:
# Validate SQL and PySpark record counts
sql_count = spark.table(
    "workspace.rearc.gold_q3_bls_population"
).count()

pyspark_count = q3_pyspark.count()

print("Q3 SQL records:", sql_count)
print("Q3 PySpark records:", pyspark_count)

assert sql_count == pyspark_count, "Record count mismatch!"

# Save PySpark implementation separately
(
    q3_pyspark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.rearc.gold_q3_bls_population_pyspark")
)

print("Q3 PYSPARK VALIDATION PASSED")
print("Q3 PYSPARK GOLD TABLE CREATED")

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.rearc
    """)
)

In [0]:
gold_tables = [
    "gold_q1_population_stats",
    "gold_q2_productivity",
    "gold_q3_bls_population",
    "gold_q3_bls_population_pyspark"
]

for table_name in gold_tables:
    count = spark.table(
        f"workspace.rearc.{table_name}"
    ).count()

    print(f"{table_name}: {count} records")

In [0]:
print("Q1 PySpark records:", q1_pyspark.count())
print("Q2 PySpark records:", q2_pyspark.count())
print("Q3 PySpark records:", q3_pyspark.count())

In [0]:
display(spark.table("workspace.rearc.gold_q1_population_stats"))

display(
    spark.table("workspace.rearc.gold_q2_productivity")
    .orderBy("series_id", "highest_year")
)

display(
    spark.table("workspace.rearc.gold_q3_bls_population")
    .orderBy("year")
)

In [0]:
print("Q2 - Sample productivity results")
display(
    spark.table("workspace.rearc.gold_q2_productivity")
    .select(
        "series_id",
        "series_label",
        "highest_year",
        "highest_quarterly_total"
    )
    .limit(10)
)

print("Q3 - Population join validation (2013–2018)")
display(
    spark.table("workspace.rearc.gold_q3_bls_population")
    .filter("year BETWEEN 2013 AND 2018")
    .select(
        "series_id",
        "year",
        "annual_value",
        "population"
    )
    .orderBy("year")
)

In [0]:
q1 = spark.table("workspace.rearc.gold_q1_population_stats")
q2 = spark.table("workspace.rearc.gold_q2_productivity")
q3 = spark.table("workspace.rearc.gold_q3_bls_population")

assert q1.first()["total_years"] == 6
assert q2.count() == 237
assert q3.count() == 39

assert q2.filter(
    "series_label IS NULL OR highest_year IS NULL"
).count() == 0

assert q3.filter(
    "year BETWEEN 2013 AND 2018 AND population IS NULL"
).count() == 0

print("ALL GOLD VALIDATIONS PASSED")